In [ ]:
import os, json, math, random, re, time
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms
from PIL import Image

from pycocotools.coco import COCO
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# ----------------------------
# Paths (as you specified)
# ----------------------------
TRAIN_IMG = Path(r"C:\COCO\train2014\train2014")
VAL_IMG   = Path(r"C:\COCO\val2017\val2017")
ANN_DIR   = Path(r"C:\COCO\annotations")
TRAIN_ANN = ANN_DIR / "captions_train2014.json"
VAL_ANN   = ANN_DIR / "captions_val2017.json"

assert TRAIN_IMG.exists(), f"Train images folder not found: {TRAIN_IMG}"
assert VAL_IMG.exists(),   f"Val images folder not found: {VAL_IMG}"
assert TRAIN_ANN.exists(), f"Train annotation not found: {TRAIN_ANN}"
assert VAL_ANN.exists(),   f"Val annotation not found: {VAL_ANN}"

# ----------------------------
# Run dir for checkpoints/logs
# ----------------------------
RUNS_ROOT = Path(r"D:\PROJECT\runs")
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = RUNS_ROOT / f"c3dc_baseline_{time.strftime('%Y%m%d_%H%M%S')}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("RUN_DIR:", RUN_DIR)


In [ ]:
SPECIALS = ["<PAD>", "<BOS>", "<EOS>", "<UNK>", "<DIV0>", "<DIV1>", "<DIV2>"]
PAD, BOS, EOS, UNK, DIV0, DIV1, DIV2 = SPECIALS

def tokenize(s: str):
    s = s.lower().strip()
    s = re.sub(r"[^a-z0-9\s]", "", s)
    return s.split()

def build_vocab(captions, min_freq=5, max_size=20000):
    counter = Counter()
    for c in captions:
        counter.update(tokenize(c))
    words = [w for w, f in counter.items() if f >= min_freq]
    words = sorted(words, key=lambda w: (-counter[w], w))[:max_size]
    itos = SPECIALS + words
    stoi = {w:i for i,w in enumerate(itos)}
    return stoi, itos

# Build vocab from TRAIN captions (COCO 2014)
coco_tr = COCO(str(TRAIN_ANN))
train_caps = [coco_tr.anns[aid]["caption"] for aid in coco_tr.anns.keys()]
stoi, itos = build_vocab(train_caps, min_freq=5, max_size=20000)

print("vocab_size:", len(itos))
print("special ids:", {s: stoi[s] for s in SPECIALS})

# Save vocab to run dir
torch.save({"stoi": stoi, "itos": itos}, RUN_DIR / "vocab.pt")
print("Saved vocab:", RUN_DIR / "vocab.pt")


In [ ]:
class CocoCaptionDataset(Dataset):
    def __init__(self, img_root, ann_json, stoi, transform, max_len=30, subset=None):
        self.img_root = Path(img_root)
        self.coco = COCO(str(ann_json))
        self.stoi = stoi
        self.transform = transform
        self.max_len = max_len

        # Flat list of (image_id, caption)
        self.items = [(ann["image_id"], ann["caption"]) for ann in self.coco.anns.values()]

        if subset is not None:
            self.items = self.items[:subset]

    def encode_caption(self, caption, div_token=DIV0):
        toks = [div_token, BOS] + tokenize(caption)[: self.max_len-2] + [EOS]
        ids = [self.stoi.get(t, self.stoi[UNK]) for t in toks]
        return torch.tensor(ids, dtype=torch.long)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        img_id, cap = self.items[idx]
        img_info = self.coco.loadImgs([img_id])[0]
        img_path = self.img_root / img_info["file_name"]

        img = Image.open(img_path).convert("RGB")
        img = self.transform(img)

        # Day-1 baseline: fixed DIV0
        cap_ids = self.encode_caption(cap, div_token=DIV0)
        return {"image": img, "cap": cap_ids}

def collate(batch, pad_id):
    imgs = torch.stack([b["image"] for b in batch], 0)
    caps = [b["cap"] for b in batch]
    maxL = max(len(c) for c in caps)
    cap_pad = torch.full((len(caps), maxL), pad_id, dtype=torch.long)
    for i,c in enumerate(caps):
        cap_pad[i,:len(c)] = c
    cap_len = torch.tensor([len(c) for c in caps], dtype=torch.long)
    return {"image": imgs, "cap": cap_pad, "cap_len": cap_len}

tfm = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

# Start with subsets to verify quickly, then set subset=None for full training
train_ds = CocoCaptionDataset(TRAIN_IMG, TRAIN_ANN, stoi, tfm, max_len=30, subset=20000)
val_ds   = CocoCaptionDataset(VAL_IMG,   VAL_ANN,   stoi, tfm, max_len=30, subset=2000)

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0,
                      collate_fn=lambda b: collate(b, stoi[PAD]))
val_dl   = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0,
                      collate_fn=lambda b: collate(b, stoi[PAD]))

print("train samples:", len(train_ds), "val samples:", len(val_ds))
print("train batches:", len(train_dl), "val batches:", len(val_dl))


In [ ]:
class ResNetEncoder(nn.Module):
    def __init__(self, d_model=512):
        super().__init__()
        base = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
        self.backbone = nn.Sequential(*list(base.children())[:-2])  # (B,2048,h,w)
        for p in self.backbone.parameters():
            p.requires_grad = False
        self.proj = nn.Conv2d(2048, d_model, 1)

    def forward(self, x):
        feat = self.backbone(x)
        feat = self.proj(feat)
        B,D,h,w = feat.shape
        tokens = feat.flatten(2).permute(2,0,1)  # (S,B,D)
        return tokens

class CaptionDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=512, nhead=8, num_layers=6, dim_ff=2048, dropout=0.2, max_len=64):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff, dropout=dropout, batch_first=False
        )
        self.dec = nn.TransformerDecoder(layer, num_layers=num_layers)
        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, tgt_ids, memory, tgt_key_padding_mask=None):
        B,T = tgt_ids.shape
        pos = torch.arange(T, device=tgt_ids.device).unsqueeze(0).expand(B,T)
        x = self.tok_emb(tgt_ids) + self.pos_emb(pos)
        x = x.permute(1,0,2)  # (T,B,D)

        causal = torch.triu(torch.ones(T,T, device=tgt_ids.device), diagonal=1).bool()
        h = self.dec(tgt=x, memory=memory, tgt_mask=causal, tgt_key_padding_mask=tgt_key_padding_mask)
        return self.out(h).permute(1,0,2)  # (B,T,V)

class CaptionModel(nn.Module):
    def __init__(self, vocab_size, d_model=512):
        super().__init__()
        self.enc = ResNetEncoder(d_model=d_model)
        self.dec = CaptionDecoder(vocab_size=vocab_size, d_model=d_model)

    def forward(self, images, cap_inp, cap_pad_mask=None):
        mem = self.enc(images)
        return self.dec(cap_inp, mem, tgt_key_padding_mask=cap_pad_mask)

def pad_mask(x, pad_id):
    return x.eq(pad_id)  # True where PAD

model = CaptionModel(vocab_size=len(itos), d_model=512).to(DEVICE)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-4, weight_decay=1e-2)

print("trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))


In [ ]:
def save_ckpt(path, model, opt, epoch, extra=None):
    ckpt = {
        "epoch": epoch,
        "model": model.state_dict(),
        "opt": opt.state_dict(),
        "extra": extra or {},
    }
    torch.save(ckpt, path)

def train_one_epoch(model, dl, opt, pad_id):
    model.train()
    total, n = 0.0, 0
    for batch in tqdm(dl, desc="train"):
        imgs = batch["image"].to(DEVICE)
        caps = batch["cap"].to(DEVICE)

        cap_in  = caps[:, :-1]
        cap_out = caps[:, 1:]
        pmask = pad_mask(cap_in, pad_id)

        logits = model(imgs, cap_in, cap_pad_mask=pmask)
        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            cap_out.reshape(-1),
            ignore_index=pad_id
        )

        opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        total += loss.item() * imgs.size(0)
        n += imgs.size(0)

    return total / max(1,n)

EPOCHS = 2  # Day-1: keep small; later increase
for ep in range(1, EPOCHS+1):
    #loss = train_one_epoch(model, train_dl, opt, pad_id=stoi[PAD])
    loss = F.cross_entropy(
    logits.reshape(-1, logits.size(-1)),
    cap_out.reshape(-1),
    ignore_index=pad_id,
    label_smoothing=0.1
    )

    print(f"Epoch {ep} train_loss={loss:.4f}")

    ckpt_path = RUN_DIR / f"checkpoint_ep{ep}.pt"
    save_ckpt(ckpt_path, model, opt, ep, extra={"train_loss": loss})
    print("Saved:", ckpt_path)

# also save a "last"
save_ckpt(RUN_DIR / "checkpoint_last.pt", model, opt, EPOCHS, extra={"note":"baseline"})
print("Saved:", RUN_DIR / "checkpoint_last.pt")


In [ ]:
@torch.no_grad()
def greedy_decode(model, image_tensor, stoi, itos, max_len=30):
    model.eval()
    img = image_tensor.unsqueeze(0).to(DEVICE)
    mem = model.enc(img)

    seq = [stoi[DIV0], stoi[BOS]]
    for _ in range(max_len):
        inp = torch.tensor(seq, device=DEVICE).unsqueeze(0)
        logits = model.dec(inp, mem, tgt_key_padding_mask=inp.eq(stoi[PAD]))[:, -1, :].squeeze(0)
        next_id = int(torch.argmax(logits).item())
        seq.append(next_id)
        if next_id == stoi[EOS]:
            break

    words = []
    for i in seq:
        tok = itos[i]
        if tok == EOS:
            break
        if tok in SPECIALS:
            continue
        words.append(tok)
    return " ".join(words)

ex = val_ds[0]
cap = greedy_decode(model, ex["image"], stoi, itos, max_len=30)
print("GREEDY:", cap)


In [ ]:
@torch.no_grad()
def decode_no_repeat(model, image_tensor, stoi, itos, max_len=30, rep_penalty=1.2, block_bigrams=True):
    model.eval()
    img = image_tensor.unsqueeze(0).to(DEVICE)
    mem = model.enc(img)

    pad_id = stoi[PAD]
    bos_id = stoi[BOS]
    eos_id = stoi[EOS]
    div_id = stoi[DIV0]
    seq = [div_id, bos_id]
    used_bigrams = set()

    for _ in range(max_len):
        inp = torch.tensor(seq, device=DEVICE).unsqueeze(0)
        logits = model.dec(inp, mem, tgt_key_padding_mask=inp.eq(pad_id))[:, -1, :].squeeze(0)

        # repetition penalty on previously used tokens
        if rep_penalty and rep_penalty > 1.0:
            for prev in set(seq):
                logits[prev] /= rep_penalty

        # bigram blocking
        if block_bigrams and len(seq) >= 2:
            prev_tok = seq[-1]
            # if we already used (prev_tok, cand) bigram, penalize candidate heavily
            for cand in range(logits.numel()):
                if (prev_tok, cand) in used_bigrams:
                    logits[cand] = -1e9

        next_id = int(torch.argmax(logits).item())
        seq.append(next_id)

        if len(seq) >= 2:
            used_bigrams.add((seq[-2], seq[-1]))

        if next_id == eos_id:
            break

    words = []
    for i in seq:
        tok = itos[i]
        if tok == EOS:
            break
        if tok in SPECIALS:
            continue
        words.append(tok)
    return " ".join(words)

# test it
ex = val_ds[0]
print("OLD GREEDY:", greedy_decode(model, ex["image"], stoi, itos))
print("NEW DECODE :", decode_no_repeat(model, ex["image"], stoi, itos))


In [ ]:
def train_one_epoch(model, dl, opt, pad_id, label_smoothing=0.1):
    model.train()
    total, n = 0.0, 0

    for batch in tqdm(dl, desc="train"):
        imgs = batch["image"].to(DEVICE)
        caps = batch["cap"].to(DEVICE)

        cap_in  = caps[:, :-1]
        cap_out = caps[:, 1:]
        pmask = pad_mask(cap_in, pad_id)

        logits = model(imgs, cap_in, cap_pad_mask=pmask)  # (B,T,V)

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            cap_out.reshape(-1),
            ignore_index=pad_id,
            label_smoothing=label_smoothing
        )

        opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        total += loss.item() * imgs.size(0)
        n += imgs.size(0)

    return total / max(1, n)


In [ ]:
import torch

CKPT_PATH = RUN_DIR / "checkpoint_last.pt"
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)

model.load_state_dict(ckpt["model"], strict=True)
opt.load_state_dict(ckpt["opt"])
start_ep = int(ckpt.get("epoch", 0))

print("Resumed from epoch:", start_ep, "=> next epoch:", start_ep+1)

MORE_EPOCHS = 5
for ep in range(start_ep+1, start_ep+MORE_EPOCHS+1):
    loss = train_one_epoch(model, train_dl, opt, pad_id=stoi[PAD])
    print(f"Epoch {ep} train_loss={loss:.4f}")

    ckpt_path = RUN_DIR / f"checkpoint_ep{ep}.pt"
    save_ckpt(ckpt_path, model, opt, ep, extra={"train_loss": loss})
    save_ckpt(RUN_DIR / "checkpoint_last.pt", model, opt, ep, extra={"train_loss": loss})
    print("Saved:", ckpt_path)


In [ ]:
import random

# pick a few random val samples
idxs = random.sample(range(len(val_ds)), 8)

for j, idx in enumerate(idxs, 1):
    ex = val_ds[idx]
    g1 = greedy_decode(model, ex["image"], stoi, itos, max_len=30)
    g2 = decode_no_repeat(model, ex["image"], stoi, itos, max_len=30, rep_penalty=1.2, block_bigrams=True)
    print(f"\n--- SAMPLE {j} (idx={idx}) ---")
    print("GREEDY      :", g1)
    print("NO-REPEAT   :", g2)



In [ ]:
class CocoCaptionDataset(Dataset):
    def __init__(self, img_root, ann_json, stoi, transform, max_len=30, subset=None):
        self.img_root = Path(img_root)
        self.coco = COCO(str(ann_json))
        self.stoi = stoi
        self.transform = transform
        self.max_len = max_len

        self.items = [(ann["image_id"], ann["caption"]) for ann in self.coco.anns.values()]
        if subset is not None:
            self.items = self.items[:subset]

    def encode_caption(self, caption, div_token):
        toks = [div_token, BOS] + tokenize(caption)[: self.max_len-2] + [EOS]
        ids = [self.stoi.get(t, self.stoi[UNK]) for t in toks]
        return torch.tensor(ids, dtype=torch.long)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        img_id, cap = self.items[idx]
        img_info = self.coco.loadImgs([img_id])[0]
        img_path = self.img_root / img_info["file_name"]

        img = Image.open(img_path).convert("RGB")
        img = self.transform(img)

        # ✅ Random diversity control token
        div = random.choice([DIV0, DIV1, DIV2])
        cap_ids = self.encode_caption(cap, div_token=div)

        return {"image": img, "cap": cap_ids}


In [ ]:
train_ds = CocoCaptionDataset(TRAIN_IMG, TRAIN_ANN, stoi, tfm, max_len=30, subset=20000)  # keep subset for now
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0,
                      collate_fn=lambda b: collate(b, stoi[PAD]))
print("train samples:", len(train_ds), "train batches:", len(train_dl))


In [ ]:
# resume from last
ckpt = torch.load(RUN_DIR / "checkpoint_last.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model"], strict=True)
opt.load_state_dict(ckpt["opt"])
start_ep = int(ckpt.get("epoch", 0))
print("Resumed at epoch:", start_ep)

MORE_EPOCHS = 3
for ep in range(start_ep+1, start_ep+MORE_EPOCHS+1):
    loss = train_one_epoch(model, train_dl, opt, pad_id=stoi[PAD], label_smoothing=0.1)
    print(f"Epoch {ep} train_loss={loss:.4f}")

    save_ckpt(RUN_DIR / f"checkpoint_ep{ep}.pt", model, opt, ep, extra={"train_loss": float(loss), "div_train": True})
    save_ckpt(RUN_DIR / "checkpoint_last.pt", model, opt, ep, extra={"train_loss": float(loss), "div_train": True})
    print("Saved ep", ep)


In [ ]:
@torch.no_grad()
def greedy_with_div(model, image_tensor, stoi, itos, div_token, max_len=30):
    model.eval()
    img = image_tensor.unsqueeze(0).to(DEVICE)
    mem = model.enc(img)

    seq = [stoi[div_token], stoi[BOS]]
    for _ in range(max_len):
        inp = torch.tensor(seq, device=DEVICE).unsqueeze(0)
        logits = model.dec(inp, mem, tgt_key_padding_mask=inp.eq(stoi[PAD]))[:, -1, :].squeeze(0)
        next_id = int(torch.argmax(logits).item())
        seq.append(next_id)
        if next_id == stoi[EOS]:
            break

    words = []
    for i in seq:
        tok = itos[i]
        if tok == EOS: break
        if tok in SPECIALS: continue
        words.append(tok)
    return " ".join(words)

ex = val_ds[0]  # any sample
print("DIV0:", greedy_with_div(model, ex["image"], stoi, itos, DIV0))
print("DIV1:", greedy_with_div(model, ex["image"], stoi, itos, DIV1))
print("DIV2:", greedy_with_div(model, ex["image"], stoi, itos, DIV2))


In [ ]:
# FULL TRAIN
train_ds = CocoCaptionDataset(TRAIN_IMG, TRAIN_ANN, stoi, tfm, max_len=30, subset=None)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0,
                      collate_fn=lambda b: collate(b, stoi[PAD]))
print("train samples:", len(train_ds), "batches:", len(train_dl))

# small VAL for monitoring (optional)
val_ds = CocoCaptionDataset(VAL_IMG, VAL_ANN, stoi, tfm, max_len=30, subset=2000)
val_dl = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0,
                    collate_fn=lambda b: collate(b, stoi[PAD]))
print("val samples:", len(val_ds), "batches:", len(val_dl))



In [ ]:
# ============================
# SWITCH TO FULL TRAIN (subset=None) + RESUME + TRAIN + CHECKPOINTS
# ============================

# 1) Recreate FULL train loader (subset=None) and small val loader (optional)
train_ds = CocoCaptionDataset(TRAIN_IMG, TRAIN_ANN, stoi, tfm, max_len=30, subset=None)
train_dl = DataLoader(
    train_ds,
    batch_size=64,
    shuffle=True,
    num_workers=0,
    collate_fn=lambda b: collate(b, stoi[PAD])
)
print("FULL train samples:", len(train_ds), "batches:", len(train_dl))

# keep val small for quick monitoring (optional)
val_ds = CocoCaptionDataset(VAL_IMG, VAL_ANN, stoi, tfm, max_len=30, subset=2000)
val_dl = DataLoader(
    val_ds,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    collate_fn=lambda b: collate(b, stoi[PAD])
)
print("VAL samples:", len(val_ds), "batches:", len(val_dl))

# 2) Resume from last checkpoint
CKPT_PATH = RUN_DIR / "checkpoint_last.pt"
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)

model.load_state_dict(ckpt["model"], strict=True)
opt.load_state_dict(ckpt["opt"])
start_ep = int(ckpt.get("epoch", 0))

print("Resumed from:", CKPT_PATH)
print("Last epoch:", start_ep, "=> next epoch:", start_ep + 1)

# 3) Train more epochs on FULL data and save checkpoints
MORE_EPOCHS = 5  # change as needed

for ep in range(start_ep + 1, start_ep + MORE_EPOCHS + 1):
    loss = train_one_epoch(model, train_dl, opt, pad_id=stoi[PAD], label_smoothing=0.1)
    print(f"[FULL] Epoch {ep} train_loss={loss:.4f}")

    # save per-epoch + last
    save_ckpt(RUN_DIR / f"checkpoint_ep{ep}.pt", model, opt, ep,
              extra={"train_loss": float(loss), "full_train": True, "div_train": True})
    save_ckpt(RUN_DIR / "checkpoint_last.pt", model, opt, ep,
              extra={"train_loss": float(loss), "full_train": True, "div_train": True})

    print("Saved:", RUN_DIR / f"checkpoint_ep{ep}.pt")


In [ ]:
# ============================
# SWITCH TO FULL TRAIN (subset=None) + RESUME + TRAIN + CHECKPOINTS
# ============================

# 1) Recreate FULL train loader (subset=None) and small val loader (optional)
train_ds = CocoCaptionDataset(TRAIN_IMG, TRAIN_ANN, stoi, tfm, max_len=30, subset=None)
train_dl = DataLoader(
    train_ds,
    batch_size=64,
    shuffle=True,
    num_workers=0,
    collate_fn=lambda b: collate(b, stoi[PAD])
)
print("FULL train samples:", len(train_ds), "batches:", len(train_dl))

# keep val small for quick monitoring (optional)
val_ds = CocoCaptionDataset(VAL_IMG, VAL_ANN, stoi, tfm, max_len=30, subset=2000)
val_dl = DataLoader(
    val_ds,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    collate_fn=lambda b: collate(b, stoi[PAD])
)
print("VAL samples:", len(val_ds), "batches:", len(val_dl))

# 2) Resume from last checkpoint
CKPT_PATH = RUN_DIR / "checkpoint_last.pt"
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)

model.load_state_dict(ckpt["model"], strict=True)
opt.load_state_dict(ckpt["opt"])
start_ep = int(ckpt.get("epoch", 0))

print("Resumed from:", CKPT_PATH)
print("Last epoch:", start_ep, "=> next epoch:", start_ep + 1)

# 3) Train more epochs on FULL data and save checkpoints
MORE_EPOCHS = 5  # change as needed

for ep in range(start_ep + 1, start_ep + MORE_EPOCHS + 1):
    loss = train_one_epoch(model, train_dl, opt, pad_id=stoi[PAD], label_smoothing=0.1)
    print(f"[FULL] Epoch {ep} train_loss={loss:.4f}")

    # save per-epoch + last
    save_ckpt(RUN_DIR / f"checkpoint_ep{ep}.pt", model, opt, ep,
              extra={"train_loss": float(loss), "full_train": True, "div_train": True})
    save_ckpt(RUN_DIR / "checkpoint_last.pt", model, opt, ep,
              extra={"train_loss": float(loss), "full_train": True, "div_train": True})

    print("Saved:", RUN_DIR / f"checkpoint_ep{ep}.pt")


In [ ]:
from pycocotools.coco import COCO
from PIL import Image
from tqdm import tqdm

from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge


In [ ]:
def eval_cider_bleu_rouge(
    N=500,
    use_div_caption="DIV0",
    max_len=30
):
    coco = COCO(str(VAL_ANN))
    img_ids = list(coco.imgs.keys())[:N]

    div_map = {"DIV0": DIV0, "DIV1": DIV1, "DIV2": DIV2}
    div_token = div_map[use_div_caption]

    gts, res = {}, {}

    for img_id in tqdm(img_ids, desc=f"VAL CIDEr@{N} ({use_div_caption})"):
        info = coco.loadImgs([img_id])[0]
        pil = Image.open(VAL_IMG / info["file_name"]).convert("RGB")
        img_t = tfm(pil)

        hyp = sample_decode(
            model, img_t, stoi, itos, div_token,
            max_len=max_len, temperature=1.0, top_k=30, top_p=0.9, rep_penalty=1.2
        )

        refs = [a["caption"] for a in coco.imgToAnns[img_id]]
        gts[img_id] = refs
        res[img_id] = [hyp]

    out = {}
    out["CIDEr"], _ = Cider().compute_score(gts, res)
    bleu, _ = Bleu(4).compute_score(gts, res)
    out["Bleu_4"] = float(bleu[3])
    out["ROUGE_L"], _ = Rouge().compute_score(gts, res)

    return {k: float(v) for k, v in out.items()}


In [ ]:
# ============================================================
# ONE-TIME RESUME CELL (First time CIDEr tracking) — start next epoch = 16
# RUN_DIR fixed as requested
# ============================================================

from pathlib import Path
import torch

RUN_DIR = Path(r"D:\PROJECT\runs\c3dc_baseline_20260210_084907")
CKPT_LAST = RUN_DIR / "checkpoint_last.pt"
CKPT_BEST = RUN_DIR / "checkpoint_best.pt"

assert CKPT_LAST.exists(), f"checkpoint_last.pt not found at: {CKPT_LAST}"
print("✅ Found:", CKPT_LAST)

# ---- load checkpoint_last.pt ----
ckpt = torch.load(CKPT_LAST, map_location=DEVICE)

# ---- restore model/optimizer ----
# (supports both formats: {"model":..., "opt":...} or raw state_dict)
if isinstance(ckpt, dict) and "model" in ckpt:
    model.load_state_dict(ckpt["model"], strict=True)
    if "opt" in ckpt and opt is not None:
        opt.load_state_dict(ckpt["opt"])
    last_epoch = int(ckpt.get("epoch", 0))
    extra = ckpt.get("extra", {}) or {}
else:
    # rare case: checkpoint is just model state_dict
    model.load_state_dict(ckpt, strict=True)
    last_epoch = 0
    extra = {}

print(f"✅ Loaded model (and optimizer if present). last_epoch(from ckpt) = {last_epoch}")

# ---- enforce your situation: ckpt is epoch 15, next should be 16 ----
# If ckpt says 15, next epoch should be 16, so start_ep=15.
# If ckpt metadata is missing/wrong, we still force start_ep=15 as per your instruction.
start_ep = 15
print(f"➡️ Will continue training from epoch {start_ep+1} (expected: 16)")

# ---- initialize best-tracking since you never computed CIDEr before ----
# If checkpoint already had these (rare), we keep them; otherwise initialize clean.
best_cider = float(extra.get("best_cider", -1e9))
best_epoch = int(extra.get("best_epoch", 0))
patience   = int(extra.get("patience", 0))

if best_cider <= -1e8:
    print("📌 No previous CIDEr tracked. Initializing best tracking fresh.")
    best_cider = -1e9
    best_epoch = 0
    patience = 0
else:
    print(f"📌 Resuming best tracking from ckpt: best_cider={best_cider:.5f} at epoch {best_epoch}, patience={patience}")

# ---- OPTIONAL: compute CIDEr@500 DIV0 NOW (first time) and save checkpoint_best.pt ----
# Set RUN_FIRST_CIDER=True if you want to establish best baseline immediately.
RUN_FIRST_CIDER = True

if RUN_FIRST_CIDER:
    metrics = eval_standard_metrics_robust(N=500, use_div_caption="DIV0")
    cur_cider = float(metrics.get("CIDEr", -1e9))
    print(f"✅ First VAL@500 DIV0 CIDEr computed: {cur_cider:.5f}")
    print(f"   Bleu4={metrics.get('Bleu_4', None)} | ROUGE_L={metrics.get('ROUGE_L', None)}")

    # Save best ckpt immediately if this is the first cider or it improves
    if cur_cider > best_cider:
        best_cider = cur_cider
        best_epoch = start_ep  # best corresponds to epoch 15 weights you just loaded
        patience = 0

        save_ckpt(CKPT_BEST, model, opt, start_ep, extra={
            "best_cider": best_cider,
            "best_epoch": best_epoch,
            "patience": patience,
            "val_metrics_best": metrics
        })
        print(f"🏁 Saved checkpoint_best.pt from epoch {start_ep} (CIDEr={best_cider:.5f})")

    # Always rewrite checkpoint_last extra so next runs remember best stats
    save_ckpt(CKPT_LAST, model, opt, start_ep, extra={
        "best_cider": best_cider,
        "best_epoch": best_epoch,
        "patience": patience,
        "val_metrics_last": metrics
    })
    print("💾 Updated checkpoint_last.pt extra stats.")


In [ ]:
# ============================================================
# ADD THIS CELL IN Untitled4 (from Untitled5): BEST CKPT + CIDEr@500 + EARLY STOP
# ============================================================

import torch
import torch.nn.functional as F
from tqdm import tqdm
from pycocotools.coco import COCO
from PIL import Image

from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider

# ----------------------------
# 1) Decoder sampler (baseline) — needed for CIDEr validation
# ----------------------------
@torch.no_grad()
def sample_decode(model, image_tensor, stoi, itos, div_token,
                  max_len=30, temperature=1.0, top_k=30, top_p=0.9,
                  rep_penalty=1.2, block_bigrams=True):
    model.eval()
    img = image_tensor.unsqueeze(0).to(DEVICE)
    mem = model.enc(img)

    pad_id = stoi["<PAD>"]
    bos_id = stoi["<BOS>"]
    eos_id = stoi["<EOS>"]
    div_id = stoi[div_token]

    seq = [div_id, bos_id]
    used_bigrams = set()

    for _ in range(max_len):
        inp = torch.tensor(seq, device=DEVICE).unsqueeze(0)
        logits = model.dec(inp, mem, tgt_key_padding_mask=inp.eq(pad_id))[:, -1, :].squeeze(0)

        # repetition penalty
        if rep_penalty and rep_penalty > 1.0:
            for prev in set(seq):
                logits[prev] /= rep_penalty

        # bigram blocking (light)
        if block_bigrams and len(seq) >= 2:
            prev_tok = seq[-1]
            # cheap but ok for COCO vocab sizes
            for cand in range(logits.numel()):
                if (prev_tok, cand) in used_bigrams:
                    logits[cand] = -1e9

        # temperature + probs
        logits = logits / max(1e-6, temperature)
        probs = F.softmax(logits, dim=-1)

        # top-k
        if top_k and top_k > 0:
            v, idx = torch.topk(probs, top_k)
            probs2 = torch.zeros_like(probs)
            probs2[idx] = v
            probs = probs2 / probs2.sum()

        # top-p
        if top_p < 1.0:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            csum = torch.cumsum(sorted_probs, dim=0)
            cut = (csum > top_p).nonzero(as_tuple=False)
            if cut.numel() > 0:
                last = cut[0].item()
                keep = sorted_idx[: last+1]
                probs2 = torch.zeros_like(probs)
                probs2[keep] = probs[keep]
                probs = probs2 / probs2.sum()

        next_id = int(torch.multinomial(probs, 1).item())
        seq.append(next_id)
        used_bigrams.add((seq[-2], seq[-1]))

        if next_id == eos_id:
            break

    # decode tokens -> string (skip specials)
    specials = {"<PAD>", "<BOS>", "<EOS>", "<UNK>", "<DIV0>", "<DIV1>", "<DIV2>"}
    words = []
    for i in seq:
        tok = itos[i]
        if tok == "<EOS>":
            break
        if tok in specials:
            continue
        words.append(tok)

    return " ".join(words)


# ----------------------------
# 2) Robust standard metrics (we only NEED CIDEr for best ckpt)
# ----------------------------
def eval_standard_metrics_robust(
    N=500,
    use_div_caption="DIV0",
    max_len=30,
    temperature=1.0,
    top_k=30,
    top_p=0.9,
    rep_penalty=1.2
):
    coco = COCO(str(VAL_ANN))
    img_ids = list(coco.imgs.keys())[:N]

    div_map = {"DIV0": DIV0, "DIV1": DIV1, "DIV2": DIV2}
    div_token = div_map[use_div_caption]

    gts, res = {}, {}

    for img_id in tqdm(img_ids, desc=f"VAL std-metrics N={N} ({use_div_caption})"):
        info = coco.loadImgs([img_id])[0]
        pil = Image.open(VAL_IMG / info["file_name"]).convert("RGB")
        img_t = tfm(pil)

        hyp = sample_decode(
            model, img_t, stoi, itos, div_token,
            max_len=max_len, temperature=temperature,
            top_k=top_k, top_p=top_p, rep_penalty=rep_penalty
        )

        refs = [a["caption"] for a in coco.imgToAnns[img_id]]
        gts[img_id] = refs
        res[img_id] = [hyp]

    scorers = [
        (Bleu(4), ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4"]),
        (Rouge(), "ROUGE_L"),
        (Cider(), "CIDEr"),
    ]

    out = {"N": N, "caption_used": use_div_caption}
    for scorer, method in scorers:
        score, _ = scorer.compute_score(gts, res)
        if isinstance(method, list):
            for s, m in zip(score, method):
                out[m] = float(s)
        else:
            out[method] = float(score)

    return out


# ----------------------------
# 3) Resume best stats (stored in checkpoint_last.pt extra)
# ----------------------------
best_path = RUN_DIR / "checkpoint_best.pt"
last_path = RUN_DIR / "checkpoint_last.pt"

best_cider = -1e9
best_epoch = 0
patience = 0

# If you already resumed model/opt earlier, this just reloads best stats from the same ckpt
if last_path.exists():
    ckpt_last = torch.load(last_path, map_location=DEVICE)
    extra = ckpt_last.get("extra", {}) or {}
    best_cider = float(extra.get("best_cider", best_cider))
    best_epoch = int(extra.get("best_epoch", best_epoch))
    patience   = int(extra.get("patience", patience))

print(f"📌 best so far: CIDEr@500={best_cider:.5f} at epoch {best_epoch} | patience={patience}")


# ----------------------------
# 4) TRAIN LOOP WITH: epoch ckpt + last ckpt + best ckpt + early stopping
# ----------------------------
EVAL_N = 500                 # paper setting
EARLY_STOP_PATIENCE = 3      # stop if no improvement for 3 epochs
MORE_EPOCHS = 3              # change to 2–3 as you decided (16–18), or any number

# IMPORTANT: start_ep should already be defined by your resume code
for ep in range(start_ep + 1, start_ep + MORE_EPOCHS + 1):
    loss = train_one_epoch(model, train_dl, opt, pad_id=stoi["<PAD>"])
    print(f"[ep {ep}] train_loss={loss:.4f}")

    # ---- save epoch checkpoint (paper reproducibility) ----
    save_ckpt(RUN_DIR / f"checkpoint_ep{ep}.pt", model, opt, ep, extra={})

    # ---- run quick validation on DIV0: CIDEr@500 ----
    metrics = eval_standard_metrics_robust(N=EVAL_N, use_div_caption="DIV0")
    cur_cider = float(metrics.get("CIDEr", -1e9))
    print(f"[ep {ep}] VAL@{EVAL_N} DIV0: CIDEr={cur_cider:.5f} | Bleu4={metrics.get('Bleu_4', None)} | ROUGE_L={metrics.get('ROUGE_L', None)}")

    # ---- best checkpoint logic ----
    improved = cur_cider > best_cider
    if improved:
        best_cider = cur_cider
        best_epoch = ep
        patience = 0

        # Save best checkpoint
        save_ckpt(best_path, model, opt, ep, extra={
            "best_cider": best_cider,
            "best_epoch": best_epoch,
            "patience": patience,
            "val_metrics_best": metrics
        })
        print(f"✅ Saved checkpoint_best.pt (epoch {ep}, CIDEr={best_cider:.5f})")
    else:
        patience += 1
        print(f"⏳ No improvement. patience={patience}/{EARLY_STOP_PATIENCE}")

    # ---- always save last checkpoint WITH best stats ----
    save_ckpt(last_path, model, opt, ep, extra={
        "best_cider": best_cider,
        "best_epoch": best_epoch,
        "patience": patience,
        "val_metrics_last": metrics
    })

    # ---- early stopping ----
    if patience >= EARLY_STOP_PATIENCE:
        print(f"🛑 Early stopping triggered. Best CIDEr@{EVAL_N}={best_cider:.5f} at epoch {best_epoch}.")
        break
